# Ploemeur single-date example

This notebook follows the same teaching structure as the synthetic example, but here the data come from a field case: there is no known truth to recover.

The goal is to understand how the observed concentrations sit in the reachable space, which parameter regions are supported by calibration, and how to interpret the Metropolis-Hastings result before comparing methods.

For a first pass, set `expert_mode = True`.

## Notebook roadmap

This notebook follows seven short steps:

1. review the dataset and model settings;
2. understand the calibration parameters;
3. inspect the observed concentrations;
4. run the calibration workflow;
5. read three beginner-friendly MH figures;
6. read the MH parameter summary;
7. inspect the expert view and the full method comparison.

In [ ]:
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import Image, Markdown, display
from IPython.utils.capture import capture_output
import matplotlib.pyplot as plt

import pyages.concentrations.concentrations as co
from pyages.lpm import build_lpm
from pyages.workflows.plots import (
    plot_objective_summary,
    plot_parameter_summary,
    plot_single_date_model_space,
)
from pyages.workflows.single_date import run_single_date
from pyages.workflows.single_date_config import load_params

ROOT = Path.cwd().resolve()
while not (ROOT / "pyages").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "pyages").exists():
    raise RuntimeError(
        "Run this notebook from the repository root or one of its subdirectories."
    )

EXAMPLE_DIR = ROOT / "examples" / "natural" / "ploemeur"


def read_tsv(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, sep="\t")
    return frame.loc[:, ~frame.columns.str.startswith("Unnamed")]


def read_stats(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t", index_col=0)


expert_mode = True

## 1. Dataset and model settings

Before running anything, it is useful to review the main settings of the field case: which dataset is used, which tracers are observed, which LPM family is calibrated, and where the YAML configuration lives.

In [ ]:
params_path = EXAMPLE_DIR / "exemple_ploemeur.yaml"
workflow_settings = yaml.safe_load(params_path.read_text(encoding="utf-8"))
params = load_params(ROOT, params_path)

dataset_path = params.dataset_data_dir / params.dataset_name
cdata = co.Concentrations(file_load=True, file_name=str(dataset_path))
tracer_names = ", ".join(cdata.cv["element"].str.upper().tolist())

case_rows = [
    {
        "parameter": "dataset label",
        "value": workflow_settings["dataset"].get("label", params.dataset_name),
    },
    {"parameter": "dataset file", "value": params.dataset_name},
    {"parameter": "reference year", "value": params.dataset_year},
    {"parameter": "data path", "value": str(dataset_path)},
    {"parameter": "tracers", "value": tracer_names},
    {"parameter": "LPM model", "value": params.lpm_model_name},
    {"parameter": "expert mode", "value": expert_mode},
]

display(pd.DataFrame(case_rows))
display(Markdown(f"Configuration file: `{params_path}`"))

## 2. Calibration parameters

The table below lists the main settings that shape the workflow. In a field case like this one, the key question is not whether the model finds a known truth, but how strongly the data constrain the feasible parameter region.

In [ ]:
calibration_specs = [
    (
        "prior",
        "run.reachable_concentrations",
        workflow_settings["run"]["reachable_concentrations"],
        "enables the pre-calibration reachable-space computation used in the first figure",
    ),
    (
        "prior",
        "reachable_concentrations.nmodels",
        1000,
        "number of sampled models used to approximate reachable space; larger means a smoother figure but a longer run",
    ),
    (
        "prior",
        "run.objective_function",
        workflow_settings["run"]["objective_function"],
        "enables the objective-function grid used in the third figure",
    ),
    (
        "prior",
        "objective_function.nmodels",
        1000,
        "size of the sampled objective grid; larger means a finer background map",
    ),
    (
        "posterior",
        "run.calibration_metropolis_hastings",
        workflow_settings["run"]["calibration_metropolis_hastings"],
        "enables the MH posterior sampling used in the beginner view",
    ),
    (
        "posterior",
        "calibration_metropolis_hastings.nstep",
        20000,
        "number of MH steps; larger means a denser posterior cloud but a longer run",
    ),
    (
        "expert comparison",
        "run.calibration_simplex",
        workflow_settings["run"]["calibration_simplex"],
        "kept enabled so that expert mode can compare MH with forward uncertainty quantification",
    ),
    (
        "expert comparison",
        "calibration_simplex.init_multiples_n",
        workflow_settings["calibration_simplex"]["init_multiples_n"],
        "controls the initial simplex multiplicity for the comparison method",
    ),
    (
        "expert comparison",
        "calibration_simplex.fuq_n",
        workflow_settings["calibration_simplex"]["fuq_n"],
        "number of forward-UQ samples used by the simplex comparison method",
    ),
]

calibration_table = pd.DataFrame(
    calibration_specs,
    columns=["phase", "parameter", "value", "role"],
)
display(calibration_table)

For a first pass, the two most useful settings to adjust are usually `calibration_metropolis_hastings.nstep` and the sampling sizes `reachable_concentrations.nmodels` / `objective_function.nmodels`.

The simplex configuration is kept in the workflow, but the notebook hides that comparison until expert mode.

## 3. Observed concentrations

This field dataset contains one sampling date and three tracers. These measured concentrations are the only direct data used by the calibration workflow.

In [ ]:
observed = cdata.cv[["element", "concentration", "error", "unit", "date"]].copy()
observed["element"] = observed["element"].str.upper()
display(observed)

## 4. Run the calibration workflow

The cell below runs the full single-date workflow. The figures are not displayed here to avoid duplicates; instead, the notebook rebuilds a simplified MH-only reading in the next section.

In [ ]:
with capture_output():
    results_dir = Path(run_single_date(params_path, force_inline=True))

display(Markdown(f"**Results directory:** `{results_dir}`"))
results_dir

## 5. Beginner view: read the MH result first

The full workflow computes both calibration methods, but the beginner reading below shows only the Metropolis-Hastings result. This keeps the first interpretation focused on one posterior cloud before any method comparison.

In [ ]:
notebook_dir = results_dir / "notebook_views"
notebook_dir.mkdir(parents=True, exist_ok=True)

mh_dist = read_tsv(results_dir / "Metropolis_Hastings" / "lpm_dist_calibrated.txt")
reachable_frame = read_tsv(results_dir / "reachable_concentrations" / "c_reach.txt")
objective_frame = read_tsv(results_dir / "objective_function_grid.txt")
param_names = build_lpm(
    params.lpm_model_name, directory_lpm=str(params.directory_lpm)
).get_param_names()
posterior_results = {"Metropolis_Hastings": mh_dist}

fig = plot_single_date_model_space(
    concentration_sampled=cdata,
    reachable_frame=reachable_frame,
    posterior_results=posterior_results,
    filename=notebook_dir / "01_mh_data_model_space.png",
    title="Ploemeur: observation, prior reachable space and MH posterior samples",
)
plt.close(fig)

fig = plot_parameter_summary(
    posterior_results,
    param_names=param_names,
    filename=notebook_dir / "02_mh_parameter_summary.png",
    title="Ploemeur: MH parameter distributions",
)
plt.close(fig)

fig = plot_objective_summary(
    objective_frame=objective_frame,
    posterior_results=posterior_results,
    param_names=param_names,
    filename=notebook_dir / "03_mh_objective_summary.png",
    title="Ploemeur: prior objective grid and MH posterior samples",
)
plt.close(fig)

In [ ]:
display(Image(filename=str(notebook_dir / "01_mh_data_model_space.png"), width=980))
display(
    Markdown("""**How to read this figure**

- the black point is the observed concentration set;
- the light background shows the reachable concentration space explored before calibration;
- the transparent blue cloud shows the MH posterior samples;
- the blue star marks the best MH sample.

This is the first question to answer in a field case: does the posterior concentrate in a narrow region of concentration space around the observation, or does it remain broad?""")
)

In [ ]:
display(Image(filename=str(notebook_dir / "02_mh_parameter_summary.png"), width=900))
display(
    Markdown("""**How to read this figure**

- each histogram shows the MH posterior distribution of one parameter;
- the center of the distribution indicates the most plausible region;
- the spread indicates how strongly the data constrain that parameter.

Because there is no known truth here, the main reading is not right versus wrong, but narrow versus broad, symmetric versus skewed, and single-mode versus multi-mode.""")
)

In [ ]:
display(Image(filename=str(notebook_dir / "03_mh_objective_summary.png"), width=980))
display(
    Markdown("""**How to read this figure**

- the colored background shows the objective function sampled on the `prior grid`;
- the blue cloud shows the MH posterior samples;
- the white star marks the best point on the prior grid;
- the blue star marks the best MH sample.

The key question is whether the MH cloud concentrates in the lowest-objective region and whether that region is narrow or broad.""")
)

## 6. Read the MH parameter summary and the modeled tracer values

After the figures, the next useful objects are the MH statistics table, the best MH sample, and a direct comparison between the tracer concentrations reproduced by that sample and the observed data.

Since there is no known truth, the goal is to summarize the center and spread of the posterior, then check how well the best calibrated model reproduces the observed CFC values.

In [ ]:
mh_stats = read_stats(results_dir / "Metropolis_Hastings" / "lpm_stats_calibrated.txt")
summary_rows = mh_stats.loc[["count", "mean", "std", "25%", "50%", "75%"]].rename(
    index={"25%": "p25", "50%": "p50", "75%": "p75"}
)
display(summary_rows)

best_sample = mh_dist.sort_values("obj_function").iloc[0]
best_row = best_sample[param_names + ["obj_function"]]
display(Markdown("### Best Metropolis-Hastings sample"))
display(best_row.to_frame("value"))

predicted_columns = cdata.names_dates()
tracer_fit = cdata.cv[["element", "concentration", "unit", "date"]].copy()
tracer_fit["element"] = tracer_fit["element"].str.upper()
tracer_fit = tracer_fit.rename(columns={"concentration": "observed_concentration"})
tracer_fit["best_mh_model_concentration"] = [
    float(best_sample[col]) for col in predicted_columns
]
tracer_fit["absolute_difference"] = (
    tracer_fit["best_mh_model_concentration"] - tracer_fit["observed_concentration"]
)
tracer_fit["relative_difference_%"] = (
    100.0 * tracer_fit["absolute_difference"] / tracer_fit["observed_concentration"]
)

display(Markdown("### Observed tracer values vs best MH model"))
display(tracer_fit.round(3))

There is no pass/fail truth test in this field case. A useful reading is instead: which parameters are tightly constrained, which ones remain broad, how different the best sample is from the center of the posterior, and whether that best sample reproduces the observed CFC values with only small residual differences.

## 7. Expert mode

Expert mode adds three things:

- an analytical MH view where posterior solutions are projected directly onto the objective surface;
- a parameter-positioning table summarizing where the best MH sample sits relative to the posterior distribution;
- the full workflow comparison between Metropolis-Hastings and forward uncertainty quantification.

Keep this section for a second pass, once the MH-only interpretation is clear.

In [ ]:
if expert_mode:
    from pyages.workflows.plots import plot_objective_solution_map

    objective_grid = read_tsv(results_dir / "objective_function_grid.txt")
    posterior = read_tsv(
        results_dir / "Metropolis_Hastings" / "lpm_dist_calibrated.txt"
    )
    posterior_stats = read_stats(
        results_dir / "Metropolis_Hastings" / "lpm_stats_calibrated.txt"
    )
    best_sample = posterior.sort_values("obj_function").iloc[0]

    plot_objective_solution_map(
        objective_frame=objective_grid,
        posterior_frame=posterior,
        param_names=param_names,
        title="Expert view: objective function and MH parameter positioning",
        cmap="viridis",  # Updated to use a more vibrant colormap
    )
    display(
        Markdown("""**How to read this expert figure**

- the interpolated background represents the objective surface sampled on the prior grid;
- the colored points are MH posterior solutions, shaded by their own objective value;
- the white star marks the best MH solution.

This view helps assess whether the MH chain really occupies the low-objective region or whether it still spreads into less favorable zones.""")
    )

    parameter_positioning = (
        pd.DataFrame(
            {
                "posterior_mean": posterior_stats.loc["mean", param_names],
                "p25": posterior_stats.loc["25%", param_names],
                "p50": posterior_stats.loc["50%", param_names],
                "p75": posterior_stats.loc["75%", param_names],
                "best_mh_sample": best_sample[param_names],
            }
        )
        .reset_index()
        .rename(columns={"index": "parameter"})
    )
    display(Markdown("### Parameter positioning relative to the posterior"))
    display(parameter_positioning.round(3))

    comparison_explanations = {
        "01_data_model_space.png": (
            "Full workflow comparison in concentration space.",
            "Compare whether MH and forward uncertainty quantification populate the same region around the observed point, or whether they suggest different feasible clouds.",
        ),
        "02_parameter_summary.png": (
            "Full workflow comparison of parameter distributions.",
            "This shows whether both methods constrain the same parameter regions or tell different stories about uncertainty.",
        ),
        "03_objective_summary.png": (
            "Full workflow comparison on the objective landscape.",
            "The key question is whether both methods identify the same low-objective region, and how concentrated each method remains around it.",
        ),
    }

    for filename, (what_it_shows, how_to_read) in comparison_explanations.items():
        figure_path = results_dir / filename
        if figure_path.exists():
            display(Markdown(f"### {filename}"))
            width = 900 if filename == "02_parameter_summary.png" else 980
            display(Image(filename=str(figure_path), width=width))
            explanation = (
                "**What it shows**\n\n"
                f"{what_it_shows}\n\n"
                "**How to read it**\n\n"
                f"{how_to_read}"
            )
            display(Markdown(explanation))
else:
    print(
        "Set expert_mode = True and re-run this cell to display the analytical and comparison views."
    )